<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 2: Relaciones Lineales</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>


Hasta este punto, hemos analizado cada variable individualmente utilizando sus distribuciones, así como medidas de forma y dispersión. Además, es posible explorar las relaciones entre dos o más variables tanto de manera gráfica como mediante estadísticos. Ahora nos enfocaremos en el estudio de la relación entre dos variables cuantitativas utilizando la covarianza, correlación y relaciones lineales que puedan existir.

In [ ]:
from sklearn.datasets import load_wine
import numpy as np
import seaborn as sns
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import pandas as pd

## Covarianza

La **covarianza** mide el grado en que dos variables varían juntas. Si ambas variables tienden a aumentar o disminuir al mismo tiempo, la covarianza será positiva. Si una aumenta mientras la otra disminuye, la covarianza será negativa. Consideremos dos conjuntos de datos $X$ y $Y$ con el mismo número de muestras: $(x_1,y_1),(x_2,y_2),...,(x_n,y_n)$ podemos calcular la covarianza Covarianza como:

$$ \mathrm{Cov}(X, Y) = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y}) $$

Donde:
- $x_i, y_i$ son los valores de las variables $X$ y $Y$.
- $\bar{x}, \bar{y}$ son las medias de $X$ y $Y$.
- $n$ es el número de observaciones.

La covarianza depende de las unidades de las variables y su magnitud no es fácil de interpretar directamente.

> Si $\mathrm{Cov}(X, Y)=0$ entonces no existe relación lineal entre $X$ e $Y$.

> Si $\mathrm{Cov}(X, Y)>0$ entonces existe una relación lineal directa o positiva entre $X$ e $Y$. Esto es, a mayores valores de $X$, en promedio tenemos mayores valores de Y y viceversa.

> Si $\mathrm{Cov}(X, Y)<0$ entonces existe una relación lineal inversa o negativa entre $X$ e $Y$. Esto es, a mayores valores de $X$, en promedio tenemos menores valores de $Y$ y viceversa.

 

#### Intuición visual

La idea detrás de la covarianza es sencilla. Para cada punto medimos **cuánto se aleja de la media** en $X$ y en $Y$, y multiplicamos ambas desviaciones:

- Si un punto está **por encima de la media en ambas** variables (o **por debajo en ambas**), el producto $(x_i-\bar{x})(y_i-\bar{y})$ es **positivo**.
- Si está **por encima en una y por debajo en la otra**, el producto es **negativo**.

La covarianza es el **promedio** de esos productos. Veámoslo con datos simulados:
</VSCode.Cell>


In [ ]:
rng = np.random.default_rng(42)

# Generamos tres relaciones: positiva, negativa y sin relación
x = rng.normal(0, 1, 200)
y_pos = x + rng.normal(0, 0.5, 200)      # varían juntas  -> Cov > 0
y_neg = -x + rng.normal(0, 0.5, 200)     # varían opuesto -> Cov < 0
y_none = rng.normal(0, 1, 200)           # independientes -> Cov ~ 0

datasets = [("Covarianza positiva", y_pos), ("Covarianza negativa", y_neg), ("Covarianza ≈ 0", y_none)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (titulo, y) in zip(axes, datasets):
    ax.scatter(x, y, s=15, alpha=0.6)
    ax.axhline(y.mean(), color='gray', ls='--', lw=1)   # media de Y
    ax.axvline(x.mean(), color='gray', ls='--', lw=1)   # media de X
    cov = np.cov(x, y)[0, 1]
    ax.set_title(f"{titulo}\nCov(X, Y) = {cov:.2f}")
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


#### Ejemplo práctico: el dataset *Iris*

Usaremos el clásico conjunto de datos **Iris**, que contiene medidas (en cm) de 150 flores de tres especies. Trabajaremos con cuatro variables numéricas: largo/ancho del **sépalo** y largo/ancho del **pétalo**.

![Partes de la flor Iris](https://raw.githubusercontent.com/scikit-learn/scikit-learn/main/doc/images/iris.svg)

>  Más información sobre el dataset: [Iris en Wikipedia](https://es.wikipedia.org/wiki/Conjunto_de_datos_flor_iris) · [documentación de scikit-learn](https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html)



In [ ]:
iris = sns.load_dataset('iris')
iris.head()

In [ ]:
iris = iris.iloc[:,:-1]

In [ ]:
# Gráfica de dispersión de petal_length vs sepal_width
plt.figure(figsize=(6,4))
plt.scatter(iris['petal_length'], iris['sepal_width'])
plt.xlabel('petal_length')
plt.ylabel('sepal_width')
plt.title('sepal_width vs petal_length')
plt.grid()


In [ ]:
cov_mat = iris.cov()
cov_mat

In [ ]:
#La covarianza entre petal_length y sepal_width es negativa; la relación lineal entre las variables es inversa.
cov_mat.loc["petal_length", "sepal_width"] 

In [ ]:
# Gráfica de dispersión de  petal_width vs petal_length
plt.figure(figsize=(6,4))
plt.scatter(iris['petal_length'],iris['petal_width'])
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.title('petal_width vs petal_length')
plt.grid()

In [ ]:
cov_mat.loc["petal_length", "petal_width"]

#### Propiedades de la covarianza
Dada una muestra $(x_1,y_1),(x_2,y_2),...,(x_n,y_n)$, se cumplen las siguientes propiedades relacionadas con la covarianza $\mathrm{Cov}(X, Y)$:

1. Si transformamos linealmente las variables originales $\hat{X}=a+bX$, $\hat{Y}=c+dY$, la covarianza $\mathrm{Cov}(\hat{X}, \hat{Y})$ es la covarianza original multiplicada por $bd$. Las constantes que se suman no alteran el resultado $\mathrm{Cov}(\hat{X}, \hat{Y}) = bd\mathrm{Cov}(X, Y)$.


In [ ]:
iris["petal_length_transform"] = 5 + 2*iris["petal_length"] # X1 = 5 + 2*X, a=5, b=2
iris["petal_width_transform"] = 3 - 4*iris["petal_width"]   # Y1 = 5 + 2*Y, c=3, d=-4

cov_mat = iris.cov()
cov_mat


In [ ]:
cov_mat.loc["petal_length", "petal_width"]

In [ ]:
cov_mat.loc["petal_length_transform", "petal_width_transform"], 2*(-4)*cov_mat.loc["petal_length", "petal_width"]

2. La covarianza de una variable consigo misma es la **varianza** de la variable: $\mathrm{Cov}(X, X) = \mathrm{Var}(X) = \sigma_X^2$


In [ ]:
iris["petal_length"].var()

In [ ]:
cov_mat.loc["petal_length", "petal_length"]

3. La covarianza entre $X$ e $Y$ es igual a la covarianza entre $Y$ y $X$:  $\mathrm{Cov}(X, Y) = \mathrm{Cov}(Y, X) $ 

In [ ]:
cov_mat.loc["petal_length", "petal_width"] == cov_mat.loc["petal_width", "petal_length"]

4. La covarianza puede calcularse también de la siguiente manera: $\mathrm{Cov}(X, Y)= \frac{\sum_{i=1}^n x_iy_i}{n-1} - \frac{n}{n-1}\bar{X}\bar{Y}$

In [ ]:
def cov_alterna(x, y): #En ocasiones, esta formulación es más sencilla de calcular que la de la definición, incluso para las computadoras.
  n = len(x)
  return ((x * y).sum() / (n - 1)) - ((n / (n - 1)) * x.mean() * y.mean())

cov_mat.loc["petal_length", "petal_width"]

In [ ]:
cov_alterna(iris["petal_length"], iris["petal_width"])

## Correlación

Aunque la covarianza nos da el signo de la relación entre dos variables, al depender de las unidades de $X$ y de $Y$, no sabemos si un valor es alto o bajo; sólo sabemos el signo. Para solucionar esto, estandarizamos los valores.

Dada una muestra $(x_1,y_1),(x_2,y_2),...,(x_n,y_n)$, calculamos la correlación entre $X$ e $Y$, y la denotamos por $r(X,Y)$ al cociente de la covarianza dividida entre el producto de las desviaciones estándar.


$$ r(X,Y) = \frac{\mathrm{Cov}(X, Y)}{\sigma_X \sigma_Y} $$

Donde:
- $\mathrm{Cov}(X, Y)$ es la covarianza entre $X$ y $Y$.
- $\sigma_X, \sigma_Y$ son las desviaciones estándar de $X$ y $Y$.

Este estadístico, también conocido como Coeficiente de correlación de Pearson se encuentra entre -1 y 1.

> $r(X,Y) = 1$: correlación positiva perfecta.

> $r(X,Y) = -1$: correlación negativa perfecta.

> $r(X,Y) = 0$: no hay relación lineal.

La correlación es adimensional y permite comparar relaciones entre variables de diferentes unidades.

#### ¿Cómo se ve cada valor de correlación?

La siguiente galería muestra nubes de puntos con **distintos valores de $r$**. Fíjate en dos cosas:

- El **signo** indica la dirección (creciente / decreciente).
- El **valor absoluto** ($|r|$) indica qué tan "apretados" están los puntos alrededor de una recta: $|r|$ cercano a 1 ⇒ relación lineal fuerte; cercano a 0 ⇒ relación lineal débil o inexistente.
</VSCode.Cell>


In [ ]:
rng = np.random.default_rng(0)

def datos_con_correlacion(r, n=300):
    """Genera n puntos (x, y) con correlación de Pearson aproximadamente r."""
    x = rng.normal(0, 1, n)
    ruido = rng.normal(0, 1, n)
    y = r * x + np.sqrt(1 - r**2) * ruido
    return x, y

valores_r = [-1.0, -0.8, -0.4, 0.0, 0.4, 0.8, 1.0]

fig, axes = plt.subplots(1, len(valores_r), figsize=(18, 3))
for ax, r in zip(axes, valores_r):
    if abs(r) == 1.0:
        x = rng.normal(0, 1, 300)
        y = r * x                      # relación perfecta: todos los puntos sobre la recta
    else:
        x, y = datos_con_correlacion(r)
    ax.scatter(x, y, s=6, alpha=0.5)
    ax.set_title(f"r = {r:.1f}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Nubes de puntos para distintos valores del coeficiente de correlación de Pearson", y=1.08)
plt.tight_layout()
plt.show()


Cuando una variable es una transformación lineal de otra, la correlación es perfecta. Por ejemplo, si tenemos dos variables, una que mide distancia recorrida en cierto tiempo y otra que mide velocidad (asumiendo que la velocidad es constante en ese mismo tiempo), el coeficiente de correlación será 1.

In [ ]:
iris.head()

In [ ]:
corr_mat = iris.corr()
corr_mat

#### Propiedades de la correlación
Dada una muestra $(x_1,y_1),(x_2,y_2),...,(x_n,y_n)$, se cumplen las siguientes propiedades relacionadas con la correlación $r(X, Y)$:

1. Si transformamos linealmente las variables originales $\hat{X}=a+bX$, $\hat{Y}=c+dY$, la correlación $r(\hat{X}, \hat{Y})$ es la correlación original multiplicada por el signo de $bd$ para cualquier $b\neq 0,, d\neq 0$.  $$r(\hat{X}, \hat{Y}) = \frac{bd}{|bd|} r(X, Y)$$


In [ ]:
iris["petal_length_transform"] = 5 + 2*iris["petal_length"]
iris["petal_width_transform"] = 3 - 4*iris["petal_width"]

corr_mat = iris.corr()
corr_mat.loc["petal_length", "petal_width"]

In [ ]:
 corr_mat.loc["petal_length", "petal_width"]*((2*-4)/(abs(2*-4)))

In [ ]:
corr_mat.loc["petal_length_transform", "petal_width_transform"]

2. La correlación de una variable consigo misma es 1

In [ ]:
corr_mat.loc["petal_length", "petal_length"]

3. La correlación entre $X$ e $Y$ es igual a la correlación entre $Y$ y $X$

In [ ]:
corr_mat.loc["petal_length", "petal_width"]

In [ ]:
corr_mat.loc["petal_width", "petal_length"]

¿Qué valor tendría la correlación entre X y −X?

$$Y = -X$$
$$r(X, Y) =  ?$$

In [ ]:
# Y = -X es una transformación lineal con b = -1 (d = -1). Por la propiedad 1,
# la correlación es r(X, X) = 1 multiplicada por el signo de bd = -1  ->  r(X, -X) = -1
X = iris["petal_length"]
Y = -X
np.corrcoef(X, Y)[0, 1]


> **Respuesta:** $r(X, -X) = -1$. Como $Y=-X$ es una transformación lineal con pendiente negativa, la relación lineal es **perfecta pero inversa**.


In [ ]:
corr = iris.corr(numeric_only=True)

# Graficar el mapa de calor de correlaciones
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de correlación del dataset Iris')
plt.show()

## ¿Cómo se ven las correlaciones?

In [ ]:
LW=load_wine()
data=LW.data
names=LW.feature_names
df=pd.DataFrame(data=data,columns=names)
df.head()

In [ ]:
wine_corr = df.corr()
wine_corr

In [ ]:
# Un mapa de calor es mucho más fácil de leer que la tabla de números anterior
plt.figure(figsize=(11, 8))
sns.heatmap(wine_corr, annot=True, cmap='coolwarm', fmt=".2f",
            vmin=-1, vmax=1, annot_kws={"size": 7})
plt.title('Matriz de correlación del dataset Wine')
plt.show()


In [ ]:
print(LW.DESCR)

## Relación entre `flavanoids` y `ash`

In [ ]:
# Gráfica flavanoids vs ash
plt.figure(figsize=(8,6))
plt.scatter(df['flavanoids'],df['ash'])
plt.xlabel('flavanoids')
plt.ylabel('ash')
plt.title('flavanoids vs ash')
plt.grid()

## Relación entre `alcalinity_of_ash` y `ash`

In [ ]:
# Gráfica alcalinity_of_ash vs ash
plt.scatter(df['alcalinity_of_ash'],df['ash'])
plt.xlabel('alcalinity_of_ash')
plt.ylabel('ash')
plt.title('alcalinity_of_ash vs ash')
plt.grid()

#### Relación lineal

Una vez que sabemos que dos variables están correlacionadas, el siguiente paso es **encontrar la mejor función lineal** que las relacione:

$$ \hat{y} = a_0 + a_1 x $$

¿Qué significa "la mejor"? La recta que **minimiza el error cuadrático medio (MSE)** entre los valores reales $y_i$ y los predichos $\hat{y}_i$:

$$ \text{MSE}(a_0, a_1) = \frac{1}{n} \sum_{i=1}^{n} \left( (a_0 + a_1 x_i) - y_i \right)^2 $$

Este método se conoce como **mínimos cuadrados ordinarios (OLS)**. A continuación lo resolvemos numéricamente con `scipy.optimize.minimize`, que busca los valores de $a_0$ y $a_1$ que hacen el MSE lo más pequeño posible.

> Lectura recomendada: [Mínimos cuadrados ordinarios (Wikipedia)](https://es.wikipedia.org/wiki/M%C3%ADnimos_cuadrados_ordinarios)


In [ ]:
minimize?

In [ ]:
# Definimos 'fun', el criterio de minimizacion
#
# ash=a[0]+a[1]*flavanoids
# y = mx + b 
# y = AX  ---->  [a0,a1]*[1, x]
def fun(a,x,y):
    f=a[0]+a[1]*x
    return np.mean((f-y)**2)

In [ ]:
a0=np.random.rand(2) # Dos elementos uniformemente distrubuidos entre 0 y 1
sol=minimize(fun,a0,args=(df.flavanoids,df.ash))
sol

In [ ]:
help(minimize)

---
La ecuación óptima que relaciona `flavanoids` y `ash` es 
$$
ash=2.3+0.0316\;flavanoids,
$$
con un $mse=0.07385$.

In [ ]:
sol.x

In [ ]:
fun(sol.x,df.flavanoids,df.ash)

In [ ]:
# Flavanoids vs ash

x=df.flavanoids
a=sol.x
f_fl_ash=a[0]+a[1]*x #ecuación de la recta
plt.figure(figsize=(5,4))
plt.scatter(df.flavanoids,df.ash,s=5) # s=size
plt.plot(x,f_fl_ash,'r') # 'r'=red
plt.xlabel('flavanoids')
plt.ylabel('ash')
plt.title(f'alcalinity_of_ash vs ash correlación: {np.round(wine_corr.loc['flavanoids', 'ash'],3)}')
plt.grid() # Cuadrícula de fondo

In [ ]:
# Función lineal que relaciona Ash vs su alcalinidad
#
# Usamos la misma función 'fun' (criterio de optimización) y las mismas condiciones iniciales 'a0'
sol2=minimize(fun,a0,args=(df.alcalinity_of_ash,df.ash))
sol2

---
La ecuación óptima que relaciona `alcalinity_of_ash` con `ash` es
$$
ash=1.656+0.0364\;alcalinity\_of\_ash,
$$
con un $mse=0.06013$.

In [ ]:
a=sol2.x
x=df.alcalinity_of_ash
f=a[0]+a[1]*x
plt.scatter(x,df.ash,s=5)
plt.plot(x,f,'r')
plt.grid()

In [ ]:
wine_corr.loc['alcalinity_of_ash', 'ash']

## Relación lineal usando Librerías

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lin = LinearRegression()

In [ ]:
df['flavanoids'].values.shape

In [ ]:
df['flavanoids'].shape

In [ ]:
df[[ 'flavanoids']].shape

In [ ]:
lin.fit(df[[ 'flavanoids']].values,df['ash'])

In [ ]:
lin.coef_

In [ ]:
lin.intercept_

In [ ]:
# y = coef_+x +intercept
f_linprog= lin.predict(df[[ 'flavanoids']])

In [ ]:
plt.scatter(df.flavanoids,df.ash,s=5) # s=size
x=df.flavanoids
#a=sol.x
#f_m=a[0]+a[1]*x #ecuación de la recta
plt.plot(x,f_fl_ash,'r') # 'r'=red
plt.plot(x,f_linprog,'k') # 'r'=black
plt.grid() # Cuadrícula de fondo

## Cuidado: lo que la correlación NO nos dice

El coeficiente de correlación de Pearson es muy útil, pero tiene **limitaciones importantes** que debes conocer para no sacar conclusiones equivocadas.

### Solo mide relaciones **lineales**

Es importante resaltar que la covarianza y el coeficiente de correlación **no detectan relaciones no lineales** entre las variables. Por ejemplo, si la relación entre $X$ e $Y$ es cuadrática (o polinómica de mayor orden), logarítmica, exponencial, etc., podríamos tener un coeficiente de correlación cercano a 0, pero esto **no significa que no haya relación** entre $X$ e $Y$; solo significa que no hay relación **lineal** entre estas variables.


In [ ]:
# Relación PERFECTAMENTE determinista pero NO lineal: una parábola y = x^2
x = np.linspace(-3, 3, 200)
y = x**2

r = np.corrcoef(x, y)[0, 1]

plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=10)
plt.title(f"y = x²  →  correlación de Pearson r = {r:.2f}\n(¡hay una relación clara, pero r ≈ 0!)")
plt.xlabel("x"); plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()


###  El cuarteto de Anscombe: **siempre grafica tus datos**

En 1973 el estadístico Francis Anscombe construyó **cuatro conjuntos de datos** que tienen prácticamente **los mismos estadísticos** (misma media, misma varianza, **misma correlación ≈ 0.816** y la misma recta de regresión)… pero que, al graficarlos, ¡son completamente distintos!

Es la mejor demostración de por qué nunca debes confiar solo en los números resumen: **siempre visualiza tus datos**.

>  [Cuarteto de Anscombe (Wikipedia)](https://es.wikipedia.org/wiki/Cuarteto_de_Anscombe) · su versión moderna y animada: [*Datasaurus Dozen*](https://www.autodesk.com/research/publications/same-stats-different-graphs)


In [ ]:
anscombe = sns.load_dataset("anscombe")

# Estadísticos por grupo: media, desviación y correlación son casi idénticos
resumen = anscombe.groupby("dataset").agg(
    media_x=("x", "mean"),
    media_y=("y", "mean"),
    std_x=("x", "std"),
    std_y=("y", "std"),
    correlacion=("x", lambda s: np.corrcoef(s, anscombe.loc[s.index, "y"])[0, 1]),
).round(2)
resumen


In [ ]:
# Mismos estadísticos, gráficas totalmente diferentes
sns.set_theme(style="ticks")
g = sns.lmplot(data=anscombe, x="x", y="y", col="dataset", hue="dataset",
               col_wrap=2, height=3.2, ci=None,
               scatter_kws={"s": 45, "alpha": 0.8}, line_kws={"color": "red"})
g.figure.suptitle("Cuarteto de Anscombe: misma recta de regresión, datos muy distintos", y=1.03)
plt.show()


### Una alternativa: la correlación de **Spearman**

Cuando la relación es **monótona** (siempre creciente o siempre decreciente) pero **no lineal**, Pearson puede subestimar la fuerza de la asociación. La **correlación de Spearman** ($\rho$) resuelve esto: calcula la correlación de Pearson sobre los **rangos** (posiciones ordenadas) de los datos en lugar de sobre los valores originales.

- **Pearson** → mide relación **lineal**.
- **Spearman** → mide relación **monótona** (aunque sea curva).

Veámoslo con una relación exponencial $y = e^{x}$:
</VSCode.Cell>


In [ ]:
from scipy.stats import pearsonr, spearmanr

x = np.linspace(0, 4, 100)
y = np.exp(x)   # relación monótona creciente, pero muy curva

r_pearson, _ = pearsonr(x, y)
r_spearman, _ = spearmanr(x, y)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=12)
plt.title(f"y = eˣ\nPearson = {r_pearson:.2f}   |   Spearman = {r_spearman:.2f}")
plt.xlabel("x"); plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()

print(f"Pearson  (relación lineal)  : {r_pearson:.3f}")
print(f"Spearman (relación monótona): {r_spearman:.3f}  <- detecta perfectamente la relación")


### Correlación ≠ Causalidad

Que dos variables estén correlacionadas **no significa que una cause la otra**. Puede haber:

- **Causalidad inversa:** $Y$ causa $X$ en lugar de $X$ causa $Y$.
- **Variable de confusión (*confounder*):** una tercera variable $Z$ causa a ambas. Ejemplo clásico: las ventas de helado y los ahogamientos están correlacionadas… pero la causa común es el **calor del verano**, no que el helado provoque ahogamientos.
- **Coincidencia (correlaciones espurias):** con suficientes variables, algunas se correlacionarán por puro azar.

> 🔗 Para reírte un rato con correlaciones absurdas pero reales: [*Spurious Correlations* de Tyler Vigen](https://www.tylervigen.com/spurious-correlations)

$$ \boxed{\text{Correlación} \;\Rightarrow\; \text{posible relación} \qquad \text{Correlación} \;\not\Rightarrow\; \text{causalidad}} $$


| Concepto | ¿Qué mide? | Rango | Clave |
|---|---|---|---|
| **Covarianza** | Dirección conjunta de la variación | $(-\infty, \infty)$ | Depende de las unidades; solo interpretable por su **signo** |
| **Correlación de Pearson** | Fuerza de la relación **lineal** | $[-1, 1]$ | Adimensional; $|r|\to 1$ ⇒ relación lineal fuerte |
| **Correlación de Spearman** | Fuerza de la relación **monótona** | $[-1, 1]$ | Usa rangos; captura relaciones curvas monótonas |
| **Regresión lineal (OLS)** | La mejor recta $\hat{y}=a_0+a_1x$ | — | Minimiza el **error cuadrático medio (MSE)** |
